# MP3 분할 — 책장 소리 제외 버전

`extract_page_turns.ipynb`에서 추출·검증한 **책장 소리 목록**(`page_turn_only_index.csv`, 347개)을 기준으로,
각 트랙을 **책장 소리가 포함되지 않는 구간**으로 잘라 트랙별 폴더에 MP3로 저장합니다.

- **입력**: `Disney Fun to Read 1/*.mp3` (23트랙), `page_turn_only_index.csv`
- **출력**: `split_no_page_turns/track01/track01_seg01_0.00-20.79s.mp3` — 트랙별 폴더 + 세그먼트 인덱스 CSV
- **규칙**: 세그먼트 = (직전 책장 소리 끝 + 여백) ~ (다음 책장 소리 시작 − 여백). 책장 소리 구간은 어느 파일에도 포함되지 않습니다.
- 트랙4의 39.2s 부근은 책장 소리가 음성과 겹쳐 있어 그 경계에서 음성 일부가 같이 잘릴 수 있습니다.
- 트랙1의 51.2s(사용자 확인상 책장 소리 아님)는 인덱스에 없으므로 그대로 유지됩니다.
- MP3 인코딩은 libsndfile 기본 설정을 사용합니다.

In [ ]:
import os, glob, csv
import numpy as np
import soundfile as sf

MP3_FOLDER = 'D:/SEMCOWork/Session26_mp3/Disney Fun to Read 1'
IDX_CSV    = 'D:/SEMCOWork/Session26_mp3/page_turn_only_index.csv'
OUT_ROOT   = 'D:/SEMCOWork/Session26_mp3/split_no_page_turns'

CFG = dict(
    guard   = 0.05,   # 책장 소리 앞뒤로 추가로 잘라내는 여백(초)
    min_seg = 0.2,    # 이보다 짧은 구간은 파일로 만들지 않음(초)
)

assert os.path.exists(IDX_CSV), '인덱스 CSV가 없습니다. 먼저 extract_page_turns.ipynb를 실행하세요.'
print('설정 완료 | guard =', CFG['guard'], 's / min_seg =', CFG['min_seg'], 's')

## 1. 책장 소리 이벤트 로드

인덱스의 `pt_start`/`pt_end`(원본 MP3 시간 기준 책장 소리 구간)를 트랙별로 모아 시간순 정렬합니다.

In [ ]:
with open(IDX_CSV, 'r', encoding='utf-8-sig') as fp:
    idx_rows = list(csv.DictReader(fp))

events = {}   # 트랙 번호 -> [(pt_start, pt_end), ...] 시간순
n_bad = 0
for r in idx_rows:
    s, e = float(r['pt_start']), float(r['pt_end'])
    if e <= s:
        n_bad += 1
        continue
    events.setdefault(int(r['track']), []).append((s, e))
for tr in events:
    events[tr].sort()

print(f'이벤트 {sum(len(v) for v in events.values())}개 / 트랙 {len(events)}개 | 비정상 구간 {n_bad}개 제외')
print('트랙1 처음 3개:', [(round(a, 2), round(b, 2)) for a, b in events[1][:3]])

## 2. 세그먼트 계산 규칙

트랙 전체에서 책장 소리 구간을 빼고, 남은 안전 구간을 세그먼트로 만듭니다.

```
[0 ──── e1.start] [e1.end ──── e2.start] … [eN.end ──── 트랙 끝]
        ↑잘림↑            ↑잘림↑
```

- 경계마다 `guard`(0.05s) 여백을 더 잘라내 책장 소리 끝단이 남지 않게 합니다.
- 두 책장 소리 사이가 `min_seg`(0.2s)보다 짧으면 파일로 만들지 않고 목록에만 기록합니다.

In [ ]:
def compute_segments(evs, total_dur, guard, min_seg):
    '''책장 소리 이벤트 사이의 안전 구간 계산.
    evs: [(start, end), ...] 시간순 / 반환: (세그먼트 목록, 짧아 제외한 구간 목록)'''
    segs, skipped, cur = [], [], 0.0
    for s, e in evs:
        a, b = cur, s - guard
        if b - a >= min_seg:
            segs.append((a, b))
        elif b - a > 0:
            skipped.append((a, b))
        cur = max(cur, e + guard)
    if total_dur - cur >= min_seg:
        segs.append((cur, total_dur))
    elif total_dur - cur > 0:
        skipped.append((cur, total_dur))
    return segs, skipped

mp3s = sorted(glob.glob(os.path.join(MP3_FOLDER, '*.mp3')))
info1 = sf.info(mp3s[0])
segs1, sk1 = compute_segments(events[1], info1.frames / info1.samplerate, CFG['guard'], CFG['min_seg'])
print(f'트랙1 미리보기: 이벤트 {len(events[1])}개 → 세그먼트 {len(segs1)}개 (짧아 제외 {len(sk1)}개)')
for a, b in segs1[:4]:
    print(f'   {a:7.2f} ~ {b:7.2f}s  ({b - a:.2f}s)')
print('   ...')

## 3. 전체 트랙 분할 실행

트랙별 폴더(`split_no_page_turns/trackNN/`)를 만들고 세그먼트를 MP3로 저장합니다.
파일명에 원본 시간 범위가 들어가 어느 부분인지 바로 알 수 있습니다.

In [ ]:
print(f'대상 파일: {len(mp3s)}개\n')

all_rows, all_skipped = [], []
for i, path in enumerate(mp3s, 1):
    if f'Set 1-{i:02d}' not in os.path.basename(path):
        print(f'  [주의] 트랙 번호 {i}와 파일명 불일치: {os.path.basename(path)}')

    info = sf.info(path)
    total = info.frames / info.samplerate
    evs = events.get(i, [])
    segs, skipped = compute_segments(evs, total, CFG['guard'], CFG['min_seg'])

    tr_dir = os.path.join(OUT_ROOT, f'track{i:02d}')
    os.makedirs(tr_dir, exist_ok=True)

    x, sr = sf.read(path)              # float, (n,) 또는 (n, ch)
    if x.ndim == 1:
        x = x[:, None]

    for k, (a, b) in enumerate(segs, 1):
        fname = f'track{i:02d}_seg{k:02d}_{a:.2f}-{b:.2f}s.mp3'
        sf.write(os.path.join(tr_dir, fname), x[int(a * sr):int(b * sr)], sr, format='MP3')
        all_rows.append(dict(track=i, seg=k, start=f'{a:.2f}', end=f'{b:.2f}',
                             dur=f'{b - a:.2f}', events=len(evs), file=fname))
    all_skipped.extend((i, a, b) for a, b in skipped)

    print(f'  track {i:2d} | 이벤트 {len(evs):2d}개 → 세그먼트 {len(segs):2d}개 | '
          f'세그먼트 합 {sum(b - a for a, b in segs):6.1f}s / 원본 {total:6.1f}s')

print(f'\n저장 위치: {OUT_ROOT}')

## 4. 결과 요약

세그먼트 목록을 `split_no_page_turns_index.csv`로 저장합니다.

In [ ]:
durs = [float(r['dur']) for r in all_rows]
print(f'세그먼트: {len(all_rows)}개 | 총 재생시간 {sum(durs) / 60:.1f}분')
print(f'길이: min={min(durs):.2f}s / median={np.median(durs):.2f}s / max={max(durs):.2f}s')
print(f'짧아 제외된 구간: {len(all_skipped)}개')
for tr, a, b in all_skipped:
    print(f'   track{tr:02d} {a:.2f}~{b:.2f}s ({b - a:.2f}s)')

out_csv = os.path.join(os.path.dirname(OUT_ROOT), 'split_no_page_turns_index.csv')
with open(out_csv, 'w', newline='', encoding='utf-8-sig') as fp:
    w = csv.DictWriter(fp, fieldnames=list(all_rows[0].keys()))
    w.writeheader()
    w.writerows(all_rows)
print(f'\n인덱스 저장: {out_csv}')

## 5. 검증

1. **구간 검증** — 모든 세그먼트가 어떤 책장 소리 구간(`pt_start − guard` ~ `pt_end + guard`)과도 겹치지 않는지 확인
2. **파일 검증** — 저장된 MP3를 무작위로 다시 읽어 길이가 인덱스와 일치하는지 확인

In [ ]:
bad = []
for r in all_rows:
    tr, a, b = int(r['track']), float(r['start']), float(r['end'])
    for s, e in events.get(tr, []):
        if a < e + CFG['guard'] - 0.01 and b > s - CFG['guard'] + 0.01:
            bad.append((r['file'], s, e))
            break
print(f'[구간 검증] 책장 소리와 겹치는 세그먼트: {len(bad)}개 →', 'PASS' if not bad else 'FAIL')
for f, s, e in bad[:5]:
    print(f'   {f} (이벤트 {s:.2f}~{e:.2f}s)')

import random
random.seed(0)
sample = random.sample(all_rows, min(5, len(all_rows)))
for r in sample:
    p = os.path.join(OUT_ROOT, f"track{int(r['track']):02d}", r['file'])
    d, srr = sf.read(p)
    assert abs(len(d) / srr - float(r['dur'])) < 0.1, p
print(f'[파일 검증] 무작위 {len(sample)}개 재읽기 길이 일치 → PASS')